# ukpyn - Official Launch Demo

**Python client for UK Power Networks Open Data**

```
pip install ukpyn
```

133+ datasets | 10 orchestrators | async-first | geospatial-ready

---
## 1. Your First Query - Three Lines of Code

No dataset IDs to memorise. No URL construction. No pagination logic.

In [ ]:
from ukpyn import ltds

# Observed peak demand at every primary substation in South East England
observed = ltds.get_table_3a(licence_area="SPN", limit=5)

print(f"Total records available: {observed.total_count}")
print(f"Returned: {len(observed.records)}")
print()

rec = observed.records[0]
for key, val in sorted(rec.fields.items()):
    print(f"  {key}: {val}")

---
## 2. Ten Orchestrators - One for Every Domain

ukpyn organises datasets into **orchestrators** — themed modules with convenience methods.

In [ ]:
from ukpyn.orchestrators import LTDSOrchestrator

orch = LTDSOrchestrator()
print("Available datasets:", orch.available_datasets)

---
## 3. Facet Discovery - See What's Inside a Dataset

Every orchestrator can show you the **facets** (filterable categories) available on any dataset. This is useful for understanding what values exist before you query.

In [ ]:
# Discover facets on the LTDS Table 3a dataset
facets = ltds.get_facets("table_3a")

print(f"Facet groups available on LTDS Table 3a:\n")
for group in facets.facets:
    print(f"  {group.name}")
    for facet in group.facets[:5]:  # show first 5 values per group
        print(f"    • {facet.name} ({facet.count:,} records)")
    if len(group.facets) > 5:
        print(f"    ... and {len(group.facets) - 5} more")
    print()

---
## 4. Facet Filtering with `refine` - Precise, Fast Queries

Once you know the facets, use `refine` to filter at the API level — faster and cleaner than SQL WHERE clauses for categorical data.

> **Heads up:** Facet names come from the raw API (e.g. `licencearea`, `fuel_type`) — but convenience methods use Pythonic parameter names (e.g. `licence_area`). The orchestrator handles the mapping for you. When using `refine={}` directly, use the raw facet names.

In [ ]:
# Step 1: See what fuel types are in the generation register
gen_facets = ltds.get_facets("table_5")

for group in gen_facets.facets:
    if group.name.lower() in ("fuel_type"):
        print(f"Fuel types in LTDS Table 5:\n")
        for f in group.facets:
            print(f"  {f.name:30s}  ({f.count:,} records)")
        break

In [ ]:
# Step 2a: Convenience method — uses Pythonic parameter names
# licence_area="SPN" maps to refine["licencearea"] = "South Eastern Power Networks (SPN)" internally
solar_gen = ltds.get_table_5(licence_area="SPN", fuel_type="CHP")

print(f"CHP sites in SPN: {solar_gen.total_count}\n")
for rec in solar_gen.records:
    f = rec.fields
    gsp = f.get("gridsupplypoint", "?")
    cap = f.get("registered_capacity_mw", f.get("installedcapacity_mva", "?"))
    fuel = f.get("fuel_type", "?")
    print(f"  {str(gsp):35s}  {str(cap):>8} MVA  ({fuel})")

In [ ]:
# Step 2b: Generic get() with raw refine — use exact facet names from get_facets()
solar_gen2 = ltds.get(
    "table_5",
    refine={"licencearea": "South Eastern Power Networks (SPN)"},
    where="fuel_type LIKE 'Photovoltaic%' OR fuel_type LIKE 'Energy Storage%'",
    limit=-1,
)

print(f"Photovoltaic + Energy Storage sites in SPN: {solar_gen2.total_count}\n")

pv = [r for r in solar_gen2.records if "Photovoltaic" in (r.fields.get("fuel_type") or "")]
es = [r for r in solar_gen2.records if "Energy Storage" in (r.fields.get("fuel_type") or "")]

for label, group in [("Photovoltaic", pv[:3]), ("Energy Storage", es[:3])]:
    print(f"  {label}:")
    for rec in group:
        f = rec.fields
        gsp = f.get("gridsupplypoint", "?")
        cap = f.get("registered_capacity_mw", f.get("installedcapacity_mva", "?"))
        print(f"    {str(gsp):35s}  {str(cap):>8} MVA")
    print()

---
## 5. Using AI

Can the project be used by AI

In [ ]:
"""
I want to use the ukpyn library to fetch the 33kv monthly timeseries of a circuit in the EPN area, convert it to a DataFrame, and create a line plot.\
Return me the code that accomplishes this task, including the DataFrame creation and the plotting steps.
"""


---
## 6. Geospatial - Every Record Has `.geometry`

All geospatial records come back with normalised GeoJSON geometry — ready for mapping.

In [ ]:
# Individual record geometries
from ukpyn import gis
substations = gis.get_primary_substations(licence_area="SPN", limit=5)

print(f"Primary substations (SPN): {substations.total_count}\n")
for rec in substations.records:
    name = rec.fields.get("sitefunctionallocation", "?")
    print(f"  {str(name):30s} → {rec.geometry}")

In [ ]:
import json

# Export entire datasets as GeoJSON — ready for QGIS, Leaflet, Mapbox, etc.
geojson_bytes = gis.export_geojson("licence_boundaries", dimensions="2d")
geojson = json.loads(geojson_bytes)

print(f"GeoJSON type: {geojson['type']}")
print(f"Features: {len(geojson['features'])}\n")

for feat in geojson["features"]:
    props = feat.get("properties", {})
    geom_type = feat["geometry"]["type"]
    print(f"  {props.get('name', '?'):30s} — {geom_type}")

# Save for use in mapping tools
with open("licence_boundaries.geojson", "w") as f:
    json.dump(geojson, f)
print("\n✓ Saved to licence_boundaries.geojson — paste into geojson.io to view")

## Round Up

- Datasets are grouped by domain in orchestrators 

- Convencience Functions to support quicker queries

- You can build your own filtering through the WHERE clause


---

## Quick Reference

| Strategy question | Key datasets | Orchestrator |
|---|---|---|
| Is there capacity for heat pumps / EVs? | `headroom`, `by_local_authority` | `dfes` |
| What's the current peak demand? | `table_3a`, `table_3b` | `ltds` |
| What generation is already connected? | `ecr`, `ecr_small`, `table_5` | `ders`, `ltds` |
| Where are the substations? | `grid_primary_sites`, `secondary_sites` | `gis` |
| What reinforcement is planned? | `projects` | `ltds` |
| Which areas are constrained? | `dnoa`, `headroom` | `dnoa`, `dfes` |
| What flex services have been dispatched? | `dispatches`, `tenders` | `flexibility` |
| Where has generation been curtailed? | `events` | `curtailment` |
| What does the network look like on a map? | `hv_overhead_lines`, `licence_boundaries` | `gis` |
| What are the half-hourly power flows? | `primary_half_hourly_*`, `132kv_half_hourly` | `powerflow` |

## Get Started

- **Docs:** https://ukpn-dso.github.io/ukpyn/
- **PyPI:** https://pypi.org/project/ukpyn/
- **Install:** `pip install ukpyn`
- **Source:** https://github.com/UKPN-DSO/ukpyn